# FinGuard: Banking Fraud Detection & Transaction Intelligence
## Notebook 02: Data Cleaning, Audit & Preprocessing Pipeline

**Objective**: Ingest raw credit card transactions, audit data quality, eliminate duplicate replay transactions, engineer temporal metadata (`Hour`, `Day`), assign unique primary keys (`Transaction_ID`), and export an investigation-ready dataset.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Configure paths
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_cleaning import (
    load_raw_data,
    inspect_data,
    check_missing_values,
    check_duplicates,
    analyze_class_distribution,
    analyze_outliers,
    clean_and_preprocess,
)

raw_path = ROOT / "data" / "raw" / "creditcard.csv"
processed_path = ROOT / "data" / "processed" / "clean_creditcard.csv"
print(f"Raw dataset path: {raw_path.resolve()}")

### 1. Ingest Raw Dataset & Schema Verification

In [ ]:
df_raw = load_raw_data(raw_path)
metadata = inspect_data(df_raw)
print(f"Rows: {metadata['num_rows']:,} | Columns: {metadata['num_columns']} | Memory: {metadata['memory_mb']:.2f} MB")
df_raw.head()

### 2. Missing Value Analysis

In [ ]:
missing_summary = check_missing_values(df_raw)
print("Total missing values across all features:", missing_summary["Missing_Count"].sum())
missing_summary.head(10)

### 3. Duplicate Transaction Identification
Duplicate transactions with identical timestamps and PCA feature vectors indicate card retries or synthetic replay noise.

In [ ]:
dup_stats = check_duplicates(df_raw)
print(f"Total duplicates found: {dup_stats['duplicate_count']:,} ({dup_stats['duplicate_percentage']:.3f}%)")
print(f"  - Legitimate duplicates: {dup_stats['legit_duplicates']:,}")
print(f"  - Fraudulent duplicates: {dup_stats['fraud_duplicates']:,}")

### 4. Class Distribution & Extreme Imbalance

In [ ]:
class_dist = analyze_class_distribution(df_raw)
print(f"Legitimate: {class_dist['legitimate_count']:,} | Fraud: {class_dist['fraud_count']:,}")
print(f"Fraud Rate: {class_dist['fraud_rate_pct']:.4f}% | Ratio: {class_dist['imbalance_ratio']}")

### 5. Monetary Outlier Analysis (Amount)

In [ ]:
outlier_stats = analyze_outliers(df_raw, column="Amount")
for k, v in outlier_stats.items():
    print(f"{k}: {v}")

### 6. Clean, Feature Engineer, and Export

In [ ]:
df_clean, audit = clean_and_preprocess(df_raw, drop_duplicates=True)
print(f"Cleaned Shape: {df_clean.shape}")
print("Engineered Columns:", [c for c in df_clean.columns if c in ["Transaction_ID", "Hour", "Day"]])
df_clean.head()